In [1]:
# 1. Indexing
from langchain_community.document_loaders import TextLoader
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_chroma import Chroma
from langchain_text_splitters import RecursiveCharacterTextSplitter

# 2. Retrieval and generation
# from langchain_community.embeddings import HuggingFaceEmbeddings # Deprecated
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_community.vectorstores import Chroma

# Constant variables
CHROMA_PATH = "chroma_db"
DATA_PATH = "data/sample.txt"

/var/folders/9g/m2fp6bjs1gs7cx_byglbc3p80000gn/T/ipykernel_79340/1009352999.py:2: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import TextLoader


In [ ]:
# Trace everything using Langsmith
#import getpass
#import os

# LangSmith: What exactly is going on inside your chain or agent
#os.environ["LANGSMITH_TRACING"] = "true"
#os.environ["LANGSMITH_API_KEY"] = getpass.getpass()

## 1. Indexing

In [2]:
# --- 1. Load document ---
print("Loading document...")
loader = TextLoader(DATA_PATH, encoding="utf-8")
documents = loader.load()
print(f"  Loaded {len(documents)} document(s)")
print(f"Document 1: {documents[0].page_content[0:80]}...")

Loading document...
  Loaded 1 document(s)
Document 1: Retrieval-Augmented Generation (RAG) is a technique that combines a retrieval sy...


In [3]:
# --- 2. Split into chunks ---
print("Splitting into chunks...")
splitter = RecursiveCharacterTextSplitter(
    chunk_size=300,
    chunk_overlap=50,
)
chunks = splitter.split_documents(documents)
print(f"  Created {len(chunks)} chunks")
for i, chunk in enumerate(chunks):
    print(f"  Chunk {i}: {chunk.page_content[:80]}...")

Splitting into chunks...
  Created 5 chunks
  Chunk 0: Retrieval-Augmented Generation (RAG) is a technique that combines a retrieval sy...
  Chunk 1: The main components of a RAG pipeline are: a document loader that reads text fro...
  Chunk 2: and searches those vectors, and a retriever that fetches the most relevant chunk...
  Chunk 3: Chroma is an open-source vector database that runs locally with no infrastructur...
  Chunk 4: LangChain is a framework for building LLM applications. It provides abstractions...


In [4]:
# --- 3. Create embeddings ---
print("Loading embedding model (first run downloads ~90MB)...")
embeddings = HuggingFaceEmbeddings(
    model_name="all-MiniLM-L6-v2"
)
embeddings

Loading embedding model (first run downloads ~90MB)...


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

HuggingFaceEmbeddings(model_name='all-MiniLM-L6-v2', cache_folder=None, model_kwargs={}, encode_kwargs={}, query_encode_kwargs={}, multi_process=False, show_progress=False)

In [5]:
# --- 4. Store in Chroma ---
print("Storing in Chroma...")
db = Chroma.from_documents(
    documents=chunks,
    embedding=embeddings,
    persist_directory=CHROMA_PATH,
)
print(f"  Stored {db._collection.count()} vectors in '{CHROMA_PATH}/'")
print("Done! Vector store ready.")

Storing in Chroma...
  Stored 5 vectors in 'chroma_db/'
Done! Vector store ready.


In [6]:
db.get(limit=100, offset=0)

{'ids': ['e40188d5-c91b-4f67-bc94-a3224957e0b0',
  '223c2676-4447-45b3-8686-0fd4a2fdc50e',
  'c368acac-dc82-4187-99c3-ccf216aa0bc7',
  '69003582-de58-4384-80be-8f438f568da6',
  'dd161fc1-6ebe-4504-951c-7d0683dcafdd'],
 'embeddings': None,
 'documents': ["Retrieval-Augmented Generation (RAG) is a technique that combines a retrieval system with a large language model. Instead of relying solely on the model's training data, RAG first searches a knowledge base for relevant documents, then passes those documents as context to the LLM.",
  'The main components of a RAG pipeline are: a document loader that reads text from files or APIs, a text splitter that breaks documents into smaller chunks, an embedding model that converts chunks into numerical vectors, a vector store that indexes and searches those vectors, and a retriever that',
  'and searches those vectors, and a retriever that fetches the most relevant chunks for a given query.',
  'Chroma is an open-source vector database that runs 

# 2. Retrieval and Generation

In [8]:
query = "What is RAG?"
results = db.similarity_search(query, k=2)

print(f"Query: '{query}'")
print(f"Top {len(results)} results:\n")
for i, doc in enumerate(results):
    print(f"--- Result {i+1} ---")
    print(doc.page_content)
    print()

Query: 'What is RAG?'
Top 2 results:

--- Result 1 ---
Retrieval-Augmented Generation (RAG) is a technique that combines a retrieval system with a large language model. Instead of relying solely on the model's training data, RAG first searches a knowledge base for relevant documents, then passes those documents as context to the LLM.

--- Result 2 ---
The main components of a RAG pipeline are: a document loader that reads text from files or APIs, a text splitter that breaks documents into smaller chunks, an embedding model that converts chunks into numerical vectors, a vector store that indexes and searches those vectors, and a retriever that



## NOTES
In the [API basic document](https://docs.langchain.com/oss/python/langchain/rag), there are two ways to use an LLM with the RAG.

**RAG chain** — the classic one (as done in this notebook):

1. The user's question comes in
2. A search is always performed on the vector store with that question
3. The retrieved chunks get pasted into the prompt
4. The LLM responds — a single call to the model

It's deterministic: question → search → response. Fast, cheap, predictable.

**RAG agent** — the modern one (multiple passes; "a chain where the LLM has control over retrieval."):

1. The question comes in
2. The LLM decides whether it needs to search or not (the search is a "tool" it can invoke)
3. It can search multiple times, reformulate the query, or not search at all if it's a "hi"
4. It responds when it has what it needs — multiple calls to the model

It's flexible but slower and less predictable: the model can decide not to search when it should.